

> Add blockquote



In [3]:
!pip install python-calamine

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 935.0/935.0 kB 10.3 MB/s  0:00:00


#  1. Đọc dữ liệu vào chương trình

In [14]:
print("Nguyễn Duy Bình - 6451071005")
import pandas as pd
import time

start = time.time()

df = pd.read_excel(
    "Online Retail.xlsx",
    engine="calamine"
)

print("Đọc dữ liệu thành công!")
print("Kích thước:", df.shape)
print("Thời gian đọc:", round(time.time() - start, 2), "giây")

display(df.head())

Nguyễn Duy Bình - 6451071005
Đọc dữ liệu thành công!
Kích thước: (541909, 8)
Thời gian đọc: 14.6 giây


,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country
0,536365,85123A,WHITE HANGING HEART T-LIGHT HOLDER,6,2010-12-01 08:26:00,2.55,17850.0,United Kingdom
1,536365,71053,WHITE METAL LANTERN,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom
2,536365,84406B,CREAM CUPID HEARTS COAT HANGER,8,2010-12-01 08:26:00,2.75,17850.0,United Kingdom
3,536365,84029G,KNITTED UNION FLAG HOT WATER BOTTLE,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom
4,536365,84029E,RED WOOLLY HOTTIE WHITE HEART.,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom


# 2. Tiền xử lý dữ liệu: Thực hiện các thao tác theo yêu cầu bài tập trước: xóa khoảng trắng thừa, loại bỏ hóa đơn thiếu và xóa hóa đơn tín dụng có ký tự C.


In [15]:
# Làm sạch tên mặt hàng
df["Description"] = df["Description"].str.strip()

# Xóa dòng thiếu số hóa đơn
df = df.dropna(subset=["InvoiceNo"]).copy()

# Chuyển số hóa đơn thành chuỗi
df["InvoiceNo"] = df["InvoiceNo"].astype(str)

# Xóa hóa đơn tín dụng
df = df[
    ~df["InvoiceNo"].str.contains("C", na=False)
].copy()

print("Kích thước sau tiền xử lý:", df.shape)

Kích thước sau tiền xử lý: (532621, 8)


# 3. Lọc dữ liệu nước Đức


In [16]:
germany = df[df["Country"] == "Germany"].copy()

print("Số dòng dữ liệu tại Đức:", len(germany))
print("Số hóa đơn tại Đức:", germany["InvoiceNo"].nunique())

display(germany.head())

Số dòng dữ liệu tại Đức: 9042
Số hóa đơn tại Đức: 457


,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country
1109,536527,22809,SET OF 6 T-LIGHTS SANTA,6,2010-12-01 13:04:00,2.95,12662.0,Germany
1110,536527,84347,ROTATING SILVER ANGELS T-LIGHT HLDR,6,2010-12-01 13:04:00,2.55,12662.0,Germany
1111,536527,84945,MULTI COLOUR SILVER T-LIGHT HOLDER,12,2010-12-01 13:04:00,0.85,12662.0,Germany
1112,536527,22242,5 HOOK HANGER MAGIC TOADSTOOL,12,2010-12-01 13:04:00,1.65,12662.0,Germany
1113,536527,22244,3 HOOK HANGER MAGIC GARDEN,12,2010-12-01 13:04:00,1.95,12662.0,Germany


# 4. Gom nhóm hóa đơn và mặt hàng

In [17]:
# Gom nhóm số lượng theo hóa đơn và mặt hàng
basket = germany.groupby(
    ["InvoiceNo", "Description"]
)["Quantity"].sum().unstack(fill_value=0)

# Chuyển sang dạng nhị phân
# 1: có mua; 0: không mua
basket = (basket > 0).astype(int)

print("Kích thước bảng giỏ hàng:", basket.shape)
display(basket.head())

Kích thước bảng giỏ hàng: (457, 1695)


Description,10 COLOUR SPACEBOY PEN,12 COLOURED PARTY BALLOONS,12 IVORY ROSE PEG PLACE SETTINGS,12 MESSAGE CARDS WITH ENVELOPES,12 PENCIL SMALL TUBE WOODLAND,12 PENCILS SMALL TUBE RED RETROSPOT,12 PENCILS SMALL TUBE SKULL,12 PENCILS TALL TUBE POSY,12 PENCILS TALL TUBE RED RETROSPOT,12 PENCILS TALL TUBE SKULLS,...,YULETIDE IMAGES GIFT WRAP SET,ZINC HEART T-LIGHT HOLDER,ZINC STAR T-LIGHT HOLDER,ZINC BOX SIGN HOME,ZINC FOLKART SLEIGH BELLS,ZINC HEART LATTICE T-LIGHT HOLDER,ZINC METAL HEART DECORATION,ZINC T-LIGHT HOLDER STAR LARGE,ZINC T-LIGHT HOLDER STARS SMALL,ZINC WILLIE WINKIE CANDLE STICK
InvoiceNo,,,,,,,,,,,,,,,,,,,,,
536527,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
536840,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
536861,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
536967,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
536983,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0


# 5. Tìm tập mặt hàng phổ biến bằng Apriori



In [18]:
from mlxtend.frequent_patterns import apriori

frequent_itemsets = apriori(
    basket,
    min_support=0.05,
    use_colnames=True
)

# Sắp xếp theo support giảm dần
frequent_itemsets = frequent_itemsets.sort_values(
    by="support",
    ascending=False
)

print("Các tập mặt hàng phổ biến:")
display(frequent_itemsets)

Các tập mặt hàng phổ biến:


/usr/local/lib/python3.13/dist-packages/mlxtend/frequent_patterns/fpcommon.py:161: DeprecationWarning: DataFrames with non-bool types result in worse computationalperformance and their support might be discontinued in the future.Please use a DataFrame with bool type
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


,support,itemsets
25,0.818381,(POSTAGE)
36,0.245077,(ROUND SNACK BOXES SET OF4 WOODLAND)
84,0.225383,"(POSTAGE, ROUND SNACK BOXES SET OF4 WOODLAND)"
35,0.157549,(ROUND SNACK BOXES SET OF 4 FRUITS)
83,0.150985,"(ROUND SNACK BOXES SET OF 4 FRUITS, POSTAGE)"
...,...,...
52,0.050328,"(POSTAGE, ALARM CLOCK BAKELIKE PINK)"
69,0.050328,"(ROUND SNACK BOXES SET OF 4 FRUITS, PLASTERS I..."
65,0.050328,"(PACK OF 20 NAPKINS RED APPLES, POSTAGE)"
81,0.050328,"(POSTAGE, RETROSPOT PARTY BAG + STICKER SET)"


/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.13/dist-packag

# Bước 6. Sinh luật kết hợp với min_conf = 50%

In [21]:
from mlxtend.frequent_patterns import association_rules

rules = association_rules(
    frequent_itemsets,
    metric="confidence",
    min_threshold=0.50
)

# Chọn các cột cần thiết
rules = rules[
    [
        "antecedents",
        "consequents",
        "support",
        "confidence",
        "lift"
    ]
].sort_values(
    by="confidence",
    ascending=False
)

print("Các luật kết hợp thỏa mãn min_conf = 50%:")
display(rules)



/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.13/dist-packag

Các luật kết hợp thỏa mãn min_conf = 50%:


/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


,antecedents,consequents,support,confidence,lift
22,(PLASTERS IN TIN STRONGMAN),(POSTAGE),0.067834,0.968750,1.183740
44,(RETROSPOT TEA SET CERAMIC 11 PC),(POSTAGE),0.054705,0.961538,1.174928
1,(ROUND SNACK BOXES SET OF 4 FRUITS),(POSTAGE),0.150985,0.958333,1.171012
5,"(ROUND SNACK BOXES SET OF 4 FRUITS, ROUND SNAC...",(POSTAGE),0.124726,0.950000,1.160829
12,(PLASTERS IN TIN SPACEBOY),(POSTAGE),0.100656,0.938776,1.147113
...,...,...,...,...,...
49,(PLASTERS IN TIN SPACEBOY),"(POSTAGE, PLASTERS IN TIN WOODLAND ANIMALS)",0.054705,0.510204,4.317838
38,"(POSTAGE, WOODLAND CHARLOTTE BAG)",(ROUND SNACK BOXES SET OF4 WOODLAND),0.059081,0.509434,2.078673
8,(ROUND SNACK BOXES SET OF4 WOODLAND),"(ROUND SNACK BOXES SET OF 4 FRUITS, POSTAGE)",0.124726,0.508929,3.370730
29,(WOODLAND CHARLOTTE BAG),(ROUND SNACK BOXES SET OF4 WOODLAND),0.063457,0.500000,2.040179


/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.13/dist-packag